# MobileNetV1（CIFAR100を用いた画像認識）

---
## 目的
MobileNetV1は，通常の畳み込みをDepthwise Separable Convolutionに置き換えることで，パラメータ数と計算量を大幅に削減しながら高い認識精度を維持する軽量なCNNである．本ノートブックでは，CIFAR100データセットを用いてMobileNetV1をスクラッチから実装し，モバイル・組み込み環境を意識した効率的なネットワーク設計について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Depthwise Separable Convolution
MobileNetV1は，2017年にGoogleから提案された，モバイル・組み込み機器での動作を想定した軽量なCNNです．通常の畳み込み層をDepthwise Separable Convolutionと呼ばれる構造に置き換えることで，認識精度を大きく落とすことなくパラメータ数・計算量を削減しています．

通常の3×3畳み込みは，空間方向のフィルタリングとチャンネル方向の混合を同時に行います．これに対しDepthwise Separable Convolutionは，この2つの処理を次の2段階に分離します．

- **Depthwise Convolution**：入力チャンネルごとに独立した1枚のフィルタを適用する畳み込み（`groups=入力チャンネル数`として実装）．チャンネル間の混合は行わず，空間方向のフィルタリングのみを行う．
- **Pointwise Convolution**：1×1の畳み込みにより，チャンネル方向の情報を混合する．

入力チャンネル数を$M$，出力チャンネル数を$N$，カーネルサイズを$D_k$とすると，通常の畳み込みの計算量は$D_k^2 M N$に比例するのに対し，Depthwise Separable Convolutionの計算量は$D_k^2 M + M N$に比例します．これにより，計算量はおよそ$\frac{1}{N} + \frac{1}{D_k^2}$倍に削減されます．

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1,
                       groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

## MobileNetV1の定義
上で定義したDepthwise Separable Convolutionを積み重ねてMobileNetV1を構築します．最初の層のみ通常の3×3畳み込みとし，以降は全てDepthwise Separable Convolutionとします．ネットワークの最後はGlobal Average Poolingと1つの全結合層のみで構成され，SqueezeNetと同様に大きな全結合層は用いません．

また，論文で提案されているwidth multiplier（$\alpha$）を`width_mult`引数として実装します．これは各層のチャンネル数に一律に乗じる係数で，$\alpha$を小さくするほどモデルサイズと計算量を削減できます（$\alpha=1.0$がベースのモデル）．

なお，オリジナルのMobileNetV1（ImageNet，224×224入力）は，stride 2の畳み込み・Depthwise Separable Convolutionを合計5回使用して特徴マップを224×224から7×7まで縮小しますが，32×32と小さいCIFAR100の入力にそのまま適用すると特徴マップがすぐに1×1以下になってしまいます．そのため，本ノートブックでは最初の畳み込みをstride 1に変更し，stride 2によるダウンサンプリングの回数を3回に減らしています（詳細は本ノートブック末尾の「ImageNet版MobileNetV1とCIFAR版MobileNetV1の違い」を参照）．

In [ ]:
class MobileNetV1(nn.Module):
    def __init__(self, n_class=100, width_mult=1.0):
        super().__init__()

        def c(channels):
            # width multiplier (alpha) に応じてチャンネル数をスケーリングする
            return max(8, int(channels * width_mult))

        def dw_sep(in_channels, out_channels, stride=1):
            return DepthwiseSeparableConv(c(in_channels), c(out_channels), stride=stride)

        self.stem = nn.Sequential(
            nn.Conv2d(3, c(32), kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c(32)),
            nn.ReLU(inplace=True),
        )

        self.layers = nn.Sequential(
            dw_sep(32, 64, stride=1),
            dw_sep(64, 128, stride=2),    # 32 -> 16
            dw_sep(128, 128, stride=1),
            dw_sep(128, 256, stride=2),   # 16 -> 8
            dw_sep(256, 256, stride=1),
            dw_sep(256, 256, stride=1),
            dw_sep(256, 256, stride=1),
            dw_sep(256, 256, stride=1),
            dw_sep(256, 256, stride=1),
            dw_sep(256, 512, stride=2),   # 8 -> 4
            dw_sep(512, 512, stride=1),
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c(512), n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したMobileNetV1を作成します．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = MobileNetV1(n_class=100, width_mult=1.0)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版MobileNetV1とCIFAR版MobileNetV1の違い
このノートブックで実装したMobileNetV1は，CIFAR100（32×32入力）に合わせてダウンサンプリング構成を変更したものです．オリジナルのMobileNetV1（ImageNet, 224×224入力）とは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 最初の畳み込み | 3×3畳み込み，stride 2 | 3×3畳み込み，stride 1 |
| stride 2によるダウンサンプリング回数 | 5回 | 3回 |
| 分類層直前の特徴マップサイズ | 7×7 | 4×4 |
| 分類層 | Global Average Pooling + 全結合層(1024→1000) | Global Average Pooling + 全結合層(512→100) |

なお，`torchvision.models`にはMobileNet V2・V3が実装されていますが，V1は含まれていない点に注意してください．

## 課題

1. `width_mult`（$\alpha$）を0.25，0.5，0.75，1.0のように変化させ，パラメータ数と認識精度のトレードオフを確認しましょう．

2. Depthwise Separable Convolutionを通常の3×3畳み込みに置き換えたネットワークを作成し，パラメータ数・学習時間・認識精度を比較しましょう．

3. stride 2を設定する層の位置やダウンサンプリングの回数を変更し，認識精度への影響を確認しましょう．